## Preparation

In [71]:
from pathlib import Path
import pandas as pd
import torch, numpy as np
from transformers import AutoModel
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from transformers import AutoTokenizer
import torch.nn as nn

# Data preparation
dataframe = pd.read_csv(str(Path.cwd().parent / 'data' / 'dataset_6.csv'))
labels = dataframe["label"].astype(int).values

# Model preparation
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert= AutoModel.from_pretrained("bert-base-uncased").eval().to("cpu")

## Extraction of CLS

In [72]:
def emb(text):
    with torch.no_grad():
        e = tok(text, return_tensors="pt", max_length=128,
                truncation=True).to(bert.device)
        return bert(**e).last_hidden_state[0,0].cpu().numpy()

X, labels = [], []
for r in tqdm(ds):
    X.append(emb(r["context"]))
    labels.append(r["label"])
X = np.vstack(X); labels = np.array(labels)

100%|██████████| 5/5 [00:00<00:00, 14.56it/s]


In [73]:
def single_iteration(current_X, target_labels):
    classifier = LogisticRegression(max_iter=1000, solver="liblinear").fit(current_X, target_labels)
    accuracy = accuracy_score(target_labels, classifier.predict(current_X))

    weight_vector = classifier.coef_[0]
    unit_vector = weight_vector / np.linalg.norm(weight_vector)

    projection_matrix = np.eye(current_X.shape[1]) - np.outer(unit_vector, unit_vector)
    projected_X = current_X @ projection_matrix.T

    return projection_matrix, projected_X, accuracy

num_iterations = 65
final_projection = np.eye(768)
X_projected = X.copy()

# Loop
for i in range(num_iterations):
    projection, X_projected, adv_accuracy = single_iteration(X_projected, labels)
    final_projection = projection @ final_projection

    print(f"[{i+1:02d}] adversarial accuracy = {adv_accuracy:.3f}")
    if adv_accuracy < 0.55:
        break

np.save("P_gender.npy", final_projection)


[01] adversarial accuracy = 1.000
[02] adversarial accuracy = 0.800
[03] adversarial accuracy = 0.600
[04] adversarial accuracy = 0.600
[05] adversarial accuracy = 0.600
[06] adversarial accuracy = 0.600
[07] adversarial accuracy = 0.600
[08] adversarial accuracy = 0.600
[09] adversarial accuracy = 0.600
[10] adversarial accuracy = 0.600
[11] adversarial accuracy = 0.600
[12] adversarial accuracy = 0.600
[13] adversarial accuracy = 0.600
[14] adversarial accuracy = 0.600
[15] adversarial accuracy = 0.600
[16] adversarial accuracy = 0.600
[17] adversarial accuracy = 0.600
[18] adversarial accuracy = 0.600
[19] adversarial accuracy = 0.600
[20] adversarial accuracy = 0.600
[21] adversarial accuracy = 0.600
[22] adversarial accuracy = 0.600
[23] adversarial accuracy = 0.600
[24] adversarial accuracy = 0.600
[25] adversarial accuracy = 0.600
[26] adversarial accuracy = 0.600
[27] adversarial accuracy = 0.600
[28] adversarial accuracy = 0.600
[29] adversarial accuracy = 0.600
[30] adversari

In [74]:
P_gender = torch.tensor(np.load("P_gender.npy")).to(bert.device)  # (768,768)

class DebiasedBert(nn.Module):
    def __init__(self, base_model, P):
        super().__init__()
        self.base = base_model
        self.register_buffer("P", P)

    def forward(self, *args, **kwargs):
        outputs = self.base(*args, **kwargs, output_hidden_states=True)
        last_h  = outputs.last_hidden_state            # (B,
        debiased = torch.matmul(last_h, self.P)        # (B, L, 768)
        return debiased, outputs

# Saves
debiased_bert = DebiasedBert(bert, P_gender)
debiased_bert.base.save_pretrained("../models/bert_inlp_gender")
tok.save_pretrained("../models/bert_inlp_gender")


('../models/bert_inlp_gender/tokenizer_config.json',
 '../models/bert_inlp_gender/special_tokens_map.json',
 '../models/bert_inlp_gender/vocab.txt',
 '../models/bert_inlp_gender/added_tokens.json',
 '../models/bert_inlp_gender/tokenizer.json')